In [28]:
import boto3
import sagemaker
import tarfile

from inference import model_fn
from sagemaker.sklearn.model import SKLearnModel

In [29]:
role = sagemaker.get_execution_role()
print(role)

arn:aws:iam::496183544310:role/LabRole


In [30]:
s3 = boto3.client("s3")
response = s3.list_buckets()

for bucket in response["Buckets"]:
    print(bucket["Name"])

sagemaker-us-east-1-496183544310


In [31]:
with tarfile.open("model.tar.gz", "w:gz") as tar:
    tar.add(
        "model/model_credit.joblib",
        arcname="model_credit.joblib"
    )

print("Model compressed successfully.")

Model compressed successfully.


In [32]:
s3 = boto3.client("s3")
bucket_name = "sagemaker-us-east-1-496183544310"

s3.upload_file(
    "model.tar.gz",
    bucket_name,
    "model/model.tar.gz"
)

print("Model uploaded to S3.")

Model uploaded to S3.


In [33]:
model = model_fn("model")
print(model)

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))]),
                                                  ['Age',
                                                   'Monthly_Inhand_Salary',
                                                   'Num_Bank_Accounts',
                                                   'Num_Credit_Card',
                                                   'Interest_Rate',
                                                   'Num_of_Loan',
                                                   'Delay_from_due_date',
                                                   'Num_of_Delayed_Payment',
                                                   'Changed_Credit_Limit',
                                                   'Num_Credit_Inquiries',
                

In [34]:
BUCKET = "sagemaker-us-east-1-496183544310"
MODEL_S3_KEY = "model/model.tar.gz"
ENDPOINT_NAME = "credit-score-endpoint"

REGION = "us-east-1"
INSTANCE_TYPE = "ml.m5.large"
FRAMEWORK_VERSION = "1.4-2"

def get_lab_role_arn():
    iam = boto3.client("iam")
    return iam.get_role(RoleName="LabRole")["Role"]["Arn"]


def main():
    boto3.setup_default_session(region_name=REGION)
    sm_session = sagemaker.Session()
    role_arn = get_lab_role_arn()
    model_s3_uri = f"s3://{BUCKET}/{MODEL_S3_KEY}"

    print(f"Role: {role_arn}")
    print(f"Model URI: {model_s3_uri}")
    print(f"Endpoint: {ENDPOINT_NAME}")

    sklearn_model = SKLearnModel(
        model_data=model_s3_uri,
        role=role_arn,
        entry_point="inference.py",
        framework_version=FRAMEWORK_VERSION,
        sagemaker_session=sm_session

    )

    predictor = sklearn_model.deploy(
        initial_instance_count=1,
        instance_type=INSTANCE_TYPE,
        endpoint_name=ENDPOINT_NAME

    )

    print("Endpoint deployed successfully.")
    
    
    runtime = boto3.client("sagemaker-runtime", region_name=REGION)

    sample = {
        "instances": [
            [
                35,
                4500,
                4,
                5,
                8,
                2,
                2,
                1,
                5,
                2,
                1200,
                25,
                180,
                700,
                3000,
                "Good",
                "No",
                "High_spent_Small_value_payments",
                120,
                11.2
            ]
        ]
    }

    response = runtime.invoke_endpoint(
        EndpointName=ENDPOINT_NAME,
        ContentType="application/json",
        Accept="application/json",
        Body=str(sample).replace("'", '"'),
    )

    print(response["Body"].read().decode("utf-8"))

if __name__ == "__main__":
    main()

Role: arn:aws:iam::496183544310:role/LabRole
Model URI: s3://sagemaker-us-east-1-496183544310/model/model.tar.gz
Endpoint: credit-score-endpoint
------!Endpoint deployed successfully.
{"prediction": "Standard", "confidence": 0.4541599345004083, "probabilities": {"Poor": 0.1685021786224847, "Standard": 0.4541599345004083, "Good": 0.3773378868771073}}


In [8]:
# check log here: https://us-east-1.console.aws.amazon.com/cloudwatch/home?utm_source=chatgpt.com&region=us-east-1#logsV2:log-groups

In [35]:
import boto3

sm_client = boto3.client("sagemaker", region_name="us-east-1")
ENDPOINT_NAME = "credit-score-endpoint"

print(f"Deleting failed endpoint: {ENDPOINT_NAME}...")
try:
    sm_client.delete_endpoint(EndpointName=ENDPOINT_NAME)
    print("Endpoint deletion triggered.")
except Exception as e:
    print(f"No endpoint found to delete: {e}")

print(f"Deleting endpoint configuration: {ENDPOINT_NAME}...")
try:
    sm_client.delete_endpoint_config(EndpointConfigName=ENDPOINT_NAME)
    print("Endpoint configuration deletion triggered.")
except Exception as e:
    print(f"No config found to delete: {e}")

print("\nCleanup complete! You can now safely run your main deploy script.")

Deleting failed endpoint: credit-score-endpoint...
Endpoint deletion triggered.
Deleting endpoint configuration: credit-score-endpoint...
Endpoint configuration deletion triggered.

Cleanup complete! You can now safely run your main deploy script.


In [37]:
sm = boto3.client('sagemaker')
print(sm.list_endpoints()['Endpoints'])

[]
